In [ ]:
import json
import struct

import torch
import whisper
from torch import nn
from whisper.model import Conv1d, LayerNorm, Linear, MultiHeadAttention, ResidualAttentionBlock

torch.set_default_dtype(torch.float16)
model = whisper.load_model('large-v3-turbo')

In [ ]:
safetensors_path = "whisper_large_v3_turbo.safetensors"
safetensors_file = open("whisper_large_v3_turbo.safetensors", "rb")
header_size_bytes = safetensors_file.read(8)
header_size = struct.unpack('Q', header_size_bytes)[0]
header_bytes = safetensors_file.read(header_size)
safetensors_data_section_offset = 8 + header_size
header_str = header_bytes.decode('utf-8')
safetensors_json = json.loads(header_str)

In [273]:
import itertools

import numpy as np
from onnx import TensorProto, numpy_helper
from onnx.checker import check_model
from onnx.helper import (
    find_min_ir_version_for,
    make_graph,
    make_model,
    make_node,
    make_opsetid,
    make_tensor_value_info,
)


def node(name: str, type: str, inputs: list[str], outputs: list[str] | None = None, **kwargs):
    if outputs is None:
        outputs = [f"{name}.out"]
    nodes.append(make_node(type, inputs, outputs, name=name, **kwargs))
    if len(outputs) == 1:
        outputs = outputs[0]
    return outputs

def initializer(name: str, ref):
    if isinstance(ref, torch.Tensor):
        for k, v in model.state_dict().items():
            if v.untyped_storage().data_ptr() == ref.untyped_storage().data_ptr():
                break
        # sadly the naming conventions are different in the hf safetensors export
        lookup_str = (
            f"model.{k}"
                .replace(".attn.", ".self_attn.")
                .replace(".cross_attn.", ".encoder_attn.")
                .replace(".key.", ".k_proj.")
                .replace(".value.", ".v_proj.")
                .replace(".query.", ".q_proj.")
                .replace(".blocks.", ".layers.")
                .replace(".token_embedding.", ".embed_tokens.")
                .replace(".positional_embedding", ".embed_positions.weight")
                .replace(".attn_ln.", ".self_attn_layer_norm.")
                .replace(".cross_attn_ln.", ".encoder_attn_layer_norm.")
                .replace(".out.", ".out_proj.")
                .replace(".mlp_ln.", ".final_layer_norm.")
                .replace(".mlp.0.", ".fc1.")
                .replace(".mlp.2.", ".fc2.")
                .replace(".ln.", ".layer_norm.")
                .replace(".ln_post.", ".layer_norm.")
        )
        entry = safetensors_json[lookup_str]
        dtype_map = {'F16': TensorProto.FLOAT16}
        dtype = dtype_map[entry["dtype"]]
        shape = entry["shape"]
        begin, end = entry["data_offsets"]
        offset = safetensors_data_section_offset + begin
        length = end - begin

        safetensors_file.seek(offset)
        buf = safetensors_file.read(length)
        ref_np = ref.detach().numpy()
        for perm in itertools.permutations(list(range(len(ref_np.shape)))):
            initial_shape = [ref_np.shape[perm.index(axis)] for axis in range(len(ref_np.shape))]
            ours = np.frombuffer(buf, dtype=np.float16).reshape(initial_shape).transpose(perm)
            if np.allclose(ours, ref_np):
                break
        else:
            raise Exception("colud not find permutation for array")

        tp = TensorProto()
        tp.name = name
        tp.dims.extend(initial_shape)
        tp.data_type = dtype
        tp.data_location = TensorProto.EXTERNAL
        entry = tp.external_data.add()
        entry.key = "location"
        entry.value = safetensors_path
        entry = tp.external_data.add()
        entry.key = "offset"
        entry.value = str(offset)
        entry = tp.external_data.add()
        entry.key = "length"
        entry.value = str(length)
        initializers.append(tp)

        if initial_shape != ref_np.shape:
            return node(f"{name}.transpose", "Transpose", [name], perm=perm)
        else:
            return name
    elif isinstance(ref, np.ndarray):
        initializers.append(numpy_helper.from_array(ref, name=name))
        return name
    else:
        raise TypeError(f"ref for initializer {name} is of unknown type")

def input(name: str, type: int, dims: list):
    inputs.append(make_tensor_value_info(name, type, dims))
    return name

def output(name: str, type: int, dims: list):
    outputs.append(make_tensor_value_info(name, type, dims))
    return name

def value_info(name: str, type: int, dims: list):
    value_infos.append(make_tensor_value_info(name, type, dims))
    return name

def linear(name: str, ref: Linear, x: str, outputs: list = None):
    weight = initializer(f'{name}.weight', ref.weight.T)
    x = node(f"{name}.mul", "MatMul", [x, weight], outputs if ref.bias is None else None)
    if ref.bias is None:
        return x
    else:
        bias = initializer(f"{name}.bias", ref.bias)
        return node(f"{name}.add", "Add", [x, bias], outputs)

def multi_head_attention(name: str, ref: MultiHeadAttention, x: str, kv: tuple[str, str] = None, kv_cache=False, causal=True, diy_attn=False):
    q = linear(f"{name}.query", ref.query, x)
    if kv is not None:
        k, v = kv
        inputs = inputs_ms = [q, k, v]
    elif kv_cache:
        k = linear(f"{name}.key", ref.key, x)
        v = linear(f"{name}.value", ref.value, x)

        past_key = input(f"past.{name}.key", TensorProto.FLOAT16, ["self_kv_cache_width", "num_heads", "past_sequence_length", "head_size"])
        past_key = node(f"{name}.past_key_gather", "Gather", [past_key, "self_kv_cache_indices"])
        past_value = input(f"past.{name}.value", TensorProto.FLOAT16, ["self_kv_cache_width", "num_heads", "past_sequence_length", "head_size"])
        past_value = node(f"{name}.past_value_gather", "Gather", [past_value, "self_kv_cache_indices"])
        inputs_ms = [q, k, v, "", "", "", past_key, past_value, self_kv_cache_lengths_maybe_int32]
        inputs = [q, k, v, "", past_key, past_value, self_kv_cache_lengths_maybe_int32]
    else:
        k = linear(f"{name}.key", ref.key, x)
        v = linear(f"{name}.value", ref.value, x)
        inputs = inputs_ms = [q, k, v]

    if not causal and not kv_cache and diy_attn:
        # we use our own DIY attention here because it seems faster?
        head_size = ref.query.bias.shape[0] // ref.n_head
        reshape_dims = initializer(f"{name}.reshape_dims", np.array([0, -1, ref.n_head, head_size]))
        q = node(f"{name}.reshape_q", "Reshape", [q, reshape_dims])
        q = node(f"{name}.transpose_q", "Transpose", [q], perm=[0, 2, 1, 3])
        k = node(f"{name}.reshape_k", "Reshape", [k, reshape_dims])
        k = node(f"{name}.transpose_k", "Transpose", [k], perm=[0, 2, 3, 1])
        v = node(f"{name}.reshape_v", "Reshape", [v, reshape_dims])
        v = node(f"{name}.transpose_v", "Transpose", [v], perm=[0, 2, 1, 3])

        scale = initializer(f"{name}.scale", np.array([1.0 / np.sqrt(head_size)], dtype=np.float16))
        q = node(f"{name}.scale_q", "Mul", [q, scale])

        qk = node(f"{name}.qk_matmul", "MatMul", [q, k], [f"{name}.attention.qk"])
        w = node(f"{name}.softmax", "Softmax", [qk])
        wv = node(f"{name}.wv_matmul", "MatMul", [w, v])
        wv = node(f"{name}.wv_transpose", "Transpose", [wv], perm=[0, 2, 1, 3])
        wv_reshape_dims = initializer(f"{name}.wv_reshape_dims", np.array([0, 0, -1]))
        wv = node(f"{name}.reshape_wv", "Reshape", [wv, wv_reshape_dims], [f"{name}.attention.wv"])
    elif use_ms_attention:
        # we dont use the default Attention block from the main ONNX domain here, because it does not have
        # a WebGPU shader yet. See docs at:
        #  https://github.com/microsoft/onnxruntime/blob/rel-1.28.0/docs/ContribOperators.md#commicrosoftmultiheadattention
        wv, present_key, present_value, qk = node(f"{name}.attention", "MultiHeadAttention", inputs_ms, [
                f"{name}.attention.wv",
                f"present.{name}.key", 
                f"present.{name}.value", 
                f"{name}.attention.qk",
            ],
            num_heads=ref.n_head, 
            unidirectional=int(causal),
            domain="com.microsoft"
            # cache_indirection is pretty cool and what we need for buffer sharing 
            # but unfortunately unsupported on webGPU as far as I (jaro) can tell on 10th Oct. 2026
        )
        value_info(qk, TensorProto.FLOAT16, 
            ["batch_size", 20, "seq_len", "dims" if kv else "total_seq_len"]
        ) # this is needed as otherwise the shape inference fails (stupid onnxruntime)
    else:
        wv, present_key, present_value, qk = node(f"{name}.attention", "Attention", inputs, [
                f"{name}.attention.wv",
                f"present.{name}.key", 
                f"present.{name}.value", 
                f"{name}.attention.qk",
            ],
            q_num_heads=ref.n_head, kv_num_heads=ref.n_head, 
            is_causal=int(causal),
        )

    if kv_cache:
        output(present_key, TensorProto.FLOAT16, ["batch_size", "num_heads", "sequence_length", "head_size"])
        output(present_value, TensorProto.FLOAT16, ["batch_size", "num_heads", "sequence_length", "head_size"])
    out = linear(f"{name}.out", ref.out, wv)
    return out

def layer_norm(name: str, ref: LayerNorm, x: str):
    # whisper does the LayerNorms in fp32 even if all other calculations are fp16
    # this is probably to help numerical stability
    weight = initializer(f"{name}.weight", ref.weight)
    # weight = node(f"{name}.weight_cast", "Cast", [weight], to=TensorProto.FLOAT)
    bias = initializer(f"{name}.bias", ref.bias)
    # bias = node(f"{name}.bias_cast", "Cast", [bias], to=TensorProto.FLOAT)
    # x = node(f"{name}.input_cast", "Cast", [x], to=TensorProto.FLOAT)
    x = node(name, "LayerNormalization", [x, weight, bias])
    # x = node(f"{name}.output_cast", "Cast", [x], to=TensorProto.FLOAT16)
    return x

def conv(name: str, ref: Conv1d, x: str, **kwargs):
    weight = initializer(f"{name}.weight", ref.weight)
    bias = initializer(f"{name}.bias", ref.bias)
    return node(name, "Conv", [x, weight, bias], **kwargs)

def mlp(name: str, ref: tuple[Linear, nn.GELU, Linear], x: str):
    x = linear(f"{name}.linear_in", ref[0], x)
    x = node(f"{name}.gelu", "Gelu", [x])
    return linear(f"{name}.linear_out", ref[2], x)

def residual_attention_block(name: str, ref: ResidualAttentionBlock, x: str, cross_kv: tuple[str, str] = None, kv_cache = True, self_attn_causal=True, diy_attn=False):
    after_ln = layer_norm(f"{name}.attn_ln", ref.attn_ln, x)
    after_attn = multi_head_attention(f"{name}.attn", ref.attn, after_ln, None, kv_cache, causal=self_attn_causal, diy_attn=diy_attn)
    x = node(f"{name}.add_attn", "Add", [x, after_attn])
    if ref.cross_attn:
        after_cross_ln = layer_norm(f"{name}.cross_attn_ln", ref.cross_attn_ln, x)
        after_cross_attn = multi_head_attention(f"{name}.cross_attn", ref.cross_attn, after_cross_ln, cross_kv, causal=False)
        x = node(f"{name}.add_cross_attn", "Add", [x, after_cross_attn])
    after_mlp_ln = layer_norm(f"{name}.mlp_ln", ref.mlp_ln, x)
    after_mlp = mlp(f"{name}.mlp", ref.mlp, after_mlp_ln)
    return node(f"{name}.add_mlp", "Add", [x, after_mlp])


In [274]:
# create cross kv cache prefill model
inputs = []
nodes = []
initializers = []
outputs = []

xa = input("encoder_hidden_states", TensorProto.FLOAT, ["batch_size", 1500, 1280])
xa = node("xa_cast", "Cast", [xa], to=TensorProto.FLOAT16)

unsqueeze_axes = initializer("unsqueeze_axes", np.array([1], dtype=np.int64))
for i, block in enumerate(model.decoder.blocks):
    k = linear(f"block.{i}.cross_attn.key", block.cross_attn.key, xa, [f"cross_attn.{i}.key"])
    output(k, TensorProto.FLOAT16, ["batch_size", 1500, 1280])
    v = linear(f"block.{i}.cross_attn.value", block.cross_attn.value, xa, [f"cross_attn.{i}.value"])
    output(v, TensorProto.FLOAT16, ["batch_size", 1500, 1280])

name = "whisper_large_v3_turbo_decoder_cross_kv_cache_prefill"
graph = make_graph(nodes, name, inputs, outputs, initializers)
onnx_model = make_model(graph)
del onnx_model.opset_import[:]
onnx_model.opset_import.extend([
    make_opsetid("", 20),
])
onnx_model.ir_version = find_min_ir_version_for(onnx_model.opset_import[:1])
check_model(onnx_model)
with open(f"{name}.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
cross_kv_cache_prefill_model = onnx_model

In [275]:
# create the rest of the decoder
nodes = []
initializers = []
inputs = []
outputs = []
value_infos = []

use_ms_attention = True

x = input('input_ids', TensorProto.INT64, ["input_seq_storage_size", "seq_len"])
seq_len = node("seq_len", "Shape", [x], start=1)

# to be used in the attention blocks (dont want to plumb it through)
indices = input('self_kv_cache_indices', TensorProto.INT64, ["batch_size"])
self_kv_cache_lengths = input("self_kv_cache_lengths", TensorProto.INT64, ["batch_size"])  # this is before reindexing through self_kv_cache_indices
self_kv_cache_lengths = node("self_kv_cache_lengths_gather", "Gather", [self_kv_cache_lengths, indices])
if use_ms_attention:
    # for stupid reasons the ms attention wants the kv cache lengths as int32
    self_kv_cache_lengths_maybe_int32 = node("self_kv_cache_lengths_cast", "Cast", [self_kv_cache_lengths], to=TensorProto.INT32) # [batch_size]
else:
    self_kv_cache_lengths_maybe_int32 = self_kv_cache_lengths


token_embedding = initializer("token_embedding", model.decoder.token_embedding.weight) # [51866, 1280]
x = node("embed_tokens", "Gather", [token_embedding, x])  # [batch_size, sequence_len, 1280]

positional_embedding = initializer("positional_embedding", model.decoder.positional_embedding) # [448, 1280]

delta = initializer("positional_embedding_range_delta", np.array([1], dtype=np.int64))
start = initializer("positional_embedding_range_start", np.array([0], dtype=np.int64))
positional_embedding_ranges = node("positional_embedding_range", "Range", [start, seq_len, delta]) # [seq_len]

unsqueeze_axes = initializer("positional_embedding_range_unsqueeze_axes", np.array([0], dtype=np.int64))
positional_embedding_ranges = node("positional_embedding_range_unsqueeze", "Unsqueeze", [positional_embedding_ranges, unsqueeze_axes]) # [1, seq_len]

unsqueeze_axes = initializer("self_kv_cache_lengths_unsqueeze_axes", np.array([1], dtype=np.int64))
lengths = node("self_kv_cache_lengths_unsqueeze", "Unsqueeze", [self_kv_cache_lengths, unsqueeze_axes]) # [batch_size, 1]

positional_embedding_ranges = node("positional_embedding_range_add", "Add", [positional_embedding_ranges, lengths]) # [batch_size, seq_len]

positional_embedding = node("positional_embedding_gather", "Gather", [positional_embedding, positional_embedding_ranges])
x = node("add_positional_embedding", "Add", [x, positional_embedding])


cross_kv_cache_indices = input('cross_kv_cache_indices', TensorProto.INT64, ["batch_size"])

squeeze_axes = initializer("kv_cache_extract_squeeze_axes", np.array([1], dtype=np.int64))
for i, block in enumerate(model.decoder.blocks):
    cross_k = input(f"cross_attn.{i}.key", TensorProto.FLOAT16, ["cross_kv_cache_size", 1500, 1280])
    cross_k = node(f"block.{i}.cross_kv_cache_gather_k", "Gather", [cross_k, cross_kv_cache_indices])
    cross_v = input(f"cross_attn.{i}.value", TensorProto.FLOAT16, ["cross_kv_cache_size", 1500, 1280])
    cross_v = node(f"block.{i}.cross_kv_cache_gather_v", "Gather", [cross_v, cross_kv_cache_indices])
    x = residual_attention_block(f"block.{i}", block, x, [cross_k, cross_v], self_attn_causal=True)
x = layer_norm("ln", model.decoder.ln, x)
token_embedding_transposed = node("transpose_token_embedding", "Transpose", [token_embedding])
x = node("matmul_final", 'MatMul', [x, token_embedding_transposed])

x = node("cast_logits", "Cast", [x], ['logits'], to=TensorProto.FLOAT)
output(x, TensorProto.FLOAT, ["batch_size" ,"input_seq_len+1", 51866])

alignment_heads_list = []
for i, (block, head) in enumerate(model.alignment_heads.indices().numpy().T):
    block_qk = f"block.{block}.cross_attn.attention.qk" # [batch_size, 20, seq_len, 1500]
    block_idx = initializer(f"alignment_head_{i}_gather_input", np.array([head], dtype=np.int64))
    head = node(f"alignment_head_{i}_gather", "Gather", [block_qk, block_idx], axis=1)
    alignment_heads_list.append(head)
alignment_heads = node("alignment_heads_concat", "Concat", alignment_heads_list, axis=1)
alignment_heads = node("alignment_heads_cast", "Cast", [alignment_heads], ["alignment_heads"], to=TensorProto.FLOAT)
output(alignment_heads, TensorProto.FLOAT, ["batch_size", len(alignment_heads_list), 1500, 1280])


name = "whisper_large_v3_turbo_decoder"
graph = make_graph(nodes, name, inputs, outputs, initializers, value_info=value_infos)
onnx_model = make_model(graph)
del onnx_model.opset_import[:]
onnx_model.opset_import.extend([
    make_opsetid("", 20),
    make_opsetid("com.microsoft", 1)
])
onnx_model.ir_version = find_min_ir_version_for(onnx_model.opset_import[:1])
check_model(onnx_model)
with open(f"{name}.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [276]:
# create the encoder
nodes = []
initializers = []
inputs = []
outputs = []
value_infos = []

use_ms_attention = True

x = input('input_features', TensorProto.FLOAT, ["batch_size", 128, 3000])
x = node("cast_input", "Cast", [x], to=TensorProto.FLOAT16)
x = conv('conv1', model.encoder.conv1, x, pads=[1, 1], kernel_shape=[3])
x = node('gelu_after_conv1', "Gelu", [x])
x = conv('conv2', model.encoder.conv2, x, pads=[1, 1], strides=[2], kernel_shape=[3])
x = node('gelu_after_conv2', "Gelu", [x])
x = node('transpose', 'Transpose', [x], perm=[0, 2, 1])

positional_embedding = initializer("positional_embedding", model.encoder.positional_embedding)
x = node('add_positional_embedding', "Add", [x, positional_embedding])

for i, block in enumerate(model.encoder.blocks):
    x = residual_attention_block(f"block.{i}", block, x, kv_cache=False, self_attn_causal=False, diy_attn=True)

x = layer_norm('ln_post', model.encoder.ln_post, x)
x = node("cast_logits", "Cast", [x], ['last_hidden_state'], to=TensorProto.FLOAT)
output(x, TensorProto.FLOAT, ["batch_size", 1500, 1280])

name = "whisper_large_v3_turbo_encoder"
graph = make_graph(nodes, name, inputs, outputs, initializers, value_info=value_infos)
onnx_model = make_model(graph)
del onnx_model.opset_import[:]
onnx_model.opset_import.extend([
    make_opsetid("", 20),
    make_opsetid("com.microsoft", 1)
])
onnx_model.ir_version = find_min_ir_version_for(onnx_model.opset_import[:1])
check_model(onnx_model)
with open(f"{name}.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [260]:
import numpy as np
from onnx.reference import ReferenceEvaluator
from onnx.model_container import ModelContainer

cross_kv_cache_prefill_model = ModelContainer()
cross_kv_cache_prefill_model.load("whisper_large_v3_turbo_decoder_cross_kv_cache_prefill.onnx")
oinf = ReferenceEvaluator(cross_kv_cache_prefill_model, verbose=3)
inputs = dict(
    encoder_hidden_states=np.array(
        np.fromfile("../../target/encoded.f32", dtype=np.float32).reshape((1, 1500, 1280)),
        dtype=np.float32,
    )
)
cross_kv_cache = oinf.run(None, inputs)


 +C unsqueeze_axes: int64:(1,) in [1, 1]
 +C block.0.cross_attn.key.weight: float16:(1280, 1280) in [-0.2445068359375, 0.30029296875]
 +C block.0.cross_attn.value.weight: float16:(1280, 1280) in [-0.048187255859375, 0.052581787109375]
 +C block.0.cross_attn.value.bias: float16:(1280,) in [-0.033843994140625, 0.0209197998046875]
 +C block.1.cross_attn.key.weight: float16:(1280, 1280) in [-0.6875, 0.49072265625]
 +C block.1.cross_attn.value.weight: float16:(1280, 1280) in [-0.0596923828125, 0.0533447265625]
 +C block.1.cross_attn.value.bias: float16:(1280,) in [-0.0208740234375, 0.0257720947265625]
 +C block.2.cross_attn.key.weight: float16:(1280, 1280) in [-0.400390625, 0.39697265625]
 +C block.2.cross_attn.value.weight: float16:(1280, 1280) in [-0.087158203125, 0.09490966796875]
 +C block.2.cross_attn.value.bias: float16:(1280,) in [-0.0355224609375, 0.034637451171875]
 +C block.3.cross_attn.key.weight: float16:(1280, 1280) in [-0.2410888671875, 0.2069091796875]
 +C block.3.cross_attn.

In [13]:
inputs = dict(
    input_ids=np.array([[
                50258,  # startoftranscript,
                50259,  # en
                50360,  # transcribe
                50364,  # notimestamps
            ]],
        dtype=np.int64),
    cross_kv_cache=cross_kv_cache,
    cross_kv_cache_indices=np.zeros((1,), dtype=np.int64),
    self_kv_cache_indices=np.array([0], dtype=np.int64),
    self_kv_cache_lengths=np.array([1], dtype=np.int64),
)
for i in range(4):
    inputs[f"past.block.{i}.attn.key"] = np.zeros((2, 20, 2, 64), np.float16)
    inputs[f"past.block.{i}.attn.value"] = np.zeros((2, 20, 2, 64), np.float16)

oinf = ReferenceEvaluator(onnx_model, verbose=3)
ours = oinf.run(None, inputs, intermediate=True)

NotImplementedError: Node type 'MultiHeadAttention' from domain 'com.microsoft' is unknown, known functions: [].

In [ ]:
inputs = dict(
    input_ids=np.array([[
        50258,  # startoftranscript,
        50259,  # en
        50360,  # transcribe
        50364,  # notimestamps
    ]], dtype=np.int64),
    encoder_hidden_states=np.array(
        np.fromfile("../../target/encoded.f32", dtype=np.float32).reshape((1, 1500, 1280)),
        dtype=np.float32,
    )
)
oinf_ref = ReferenceEvaluator("decoder_model_ref_fp16.onnx", verbose=3)
theirs = oinf_ref.run(None, inputs, intermediate=True)

 +C model.decoder.embed_tokens.weight: float16:(51866, 1280) in [-0.244384765625, 0.10986328125]
 +C /model/decoder/Constant_output_0: int64:() in [0, 0]
 +C /model/decoder/Constant_1_output_0: int64:() in [1, 1]
 +C onnx::MatMul_1820: float16:(1280, 1280) in [-0.2445068359375, 0.30029296875]
 +C onnx::MatMul_1827: float16:(1280, 1280) in [-0.048187255859375, 0.052581787109375]
 +C onnx::MatMul_1865: float16:(1280, 1280) in [-0.6875, 0.49072265625]
 +C onnx::MatMul_1872: float16:(1280, 1280) in [-0.0596923828125, 0.0533447265625]
 +C onnx::MatMul_1910: float16:(1280, 1280) in [-0.400390625, 0.39697265625]
 +C onnx::MatMul_1917: float16:(1280, 1280) in [-0.087158203125, 0.09490966796875]
 +C onnx::MatMul_1955: float16:(1280, 1280) in [-0.2410888671875, 0.2069091796875]
 +C onnx::MatMul_1962: float16:(1280, 1280) in [-0.1588134765625, 0.1585693359375]
 +C onnx::Unsqueeze_600: int64:(1,) in [0, 0]
 +C /model/decoder/Constant_18_output_0: int64:(1,) in [0, 0]
 +C model.decoder.layers.0.enc

In [ ]:
np.testing.assert_allclose(ours["add_positional_embedding.out"], theirs["/model/decoder/Add_2_output_0"])
for i in range(4):
    np.testing.assert_allclose(ours[f"block.{i}.cross_kv_cache_squeeze_k.out"], theirs[f"/model/decoder/layers.{i}/encoder_attn/k_proj/MatMul_output_0"], atol=0.001)
    np.testing.assert_allclose(ours[f"block.{i}.cross_kv_cache_squeeze_v.out"], theirs[f"/model/decoder/layers.{i}/encoder_attn/v_proj/Add_output_0"], atol=0.001)
np.testing.assert_allclose(ours["block.0.attn_ln.out"], theirs["/model/decoder/layers.0/self_attn_layer_norm/Add_1_output_0"], atol=0.001)
np.testing.assert_allclose(ours["block.0.attn.query.add.out"], theirs["/model/decoder/layers.0/self_attn/q_proj/Add_output_0"], atol=0.001)
np.testing.assert_allclose(ours["block.0.attn.value.add.out"], theirs["/model/decoder/layers.0/self_attn/v_proj/Add_output_0"], atol=0.001)
np.testing.assert_allclose(ours["block.0.attn.key.mul.out"], theirs["/model/decoder/layers.0/self_attn/k_proj/MatMul_output_0"], atol=0.002)
np.testing.assert_allclose(ours["block.0.attn.attention.wv"], theirs["/model/decoder/layers.0/self_attn/Reshape_9_output_0"])
assert np.allclose(ours["logits"], theirs["logits"], atol=1e-7)

In [11]:
import numpy as np
from onnx.reference import ReferenceEvaluator

oinf = ReferenceEvaluator(onnx_model, verbose=3)
inputs = dict(
    input_features=np.zeros((1, 128, 3000), dtype=np.float32)
)
ours = oinf.run(None, inputs, intermediate=True)


 +C conv1.weight: float32:(1280, 128, 3) in [-0.27783203125, 0.348388671875]
 +C conv1.bias: float32:(1280,) in [-0.30419921875, 0.251220703125]
 +C conv2.weight: float32:(1280, 1280, 3) in [-0.3125, 0.3193359375]
 +C conv2.bias: float32:(1280,) in [-0.06280517578125, 0.11572265625]
 +C positional_embedding: float32:(1500, 1280) in [-1.0, 1.0]
 +C block.0.attn_ln.weight: float32:(1280,) in [-0.3095703125, 1.255859375]
 +C block.0.attn_ln.bias: float32:(1280,) in [-0.055023193359375, 0.09429931640625]
 +C block.0.attn.query.weight: float32:(1280, 1280) in [-0.619140625, 0.63720703125]
 +C block.0.attn.query.bias: float32:(1280,) in [-0.1746826171875, 0.1729736328125]
 +C block.0.attn.key.weight: float32:(1280, 1280) in [-0.64892578125, 0.6025390625]
 +C block.0.attn.value.weight: float32:(1280, 1280) in [-0.287841796875, 0.2393798828125]
 +C block.0.attn.value.bias: float32:(1280,) in [-0.045654296875, 0.04705810546875]
 +C block.0.attn.reshape_dims: int64:(4,) in [-1, 64]
 +C block.0.a

In [13]:
oinf = ReferenceEvaluator("encoder_model.onnx", verbose=3)
inputs = dict(
    input_features=np.zeros((1, 128, 3000), dtype=np.float32)
)
theirs = oinf.run(None, inputs, intermediate=True)


 +C conv1.weight: float32:(1280, 128, 3) in [-0.27783203125, 0.348388671875]
 +C conv1.bias: float32:(1280,) in [-0.30419921875, 0.251220703125]
 +C /Constant_output_0: float32:() in [1.4142135381698608, 1.4142135381698608]
 +C /Constant_1_output_0: float32:() in [1.0, 1.0]
 +C /Constant_2_output_0: float32:() in [0.5, 0.5]
 +C conv2.weight: float32:(1280, 1280, 3) in [-0.3125, 0.3193359375]
 +C conv2.bias: float32:(1280,) in [-0.06280517578125, 0.11572265625]
 +C /Constant_3_output_0: float32:() in [1.4142135381698608, 1.4142135381698608]
 +C /Constant_4_output_0: float32:() in [1.0, 1.0]
 +C /Constant_5_output_0: float32:() in [0.5, 0.5]
 +C embed_positions.weight: float32:(1500, 1280) in [-1.0, 1.0]
 +C /layers.0/self_attn_layer_norm/Constant_output_0: float32:() in [2.0, 2.0]
 +C /layers.0/self_attn_layer_norm/Constant_1_output_0: float32:() in [9.999999747378752e-06, 9.999999747378752e-06]
 +C layers.0.self_attn_layer_norm.weight: float32:(1280,) in [-0.3095703125, 1.255859375]
 +

In [27]:
np.testing.assert_allclose(ours["block.0.attn.scale_k.out"], theirs["/layers.0/self_attn/Transpose_2_output_0"])

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=0

Mismatched elements: 1920000 / 1920000 (100%)
First 5 mismatches are at indices:
 [0, 0, 0, 0]: 0.01739010028541088 (ACTUAL), 0.13912074267864227 (DESIRED)
 [0, 0, 0, 1]: 0.032302699983119965 (ACTUAL), 0.2584216296672821 (DESIRED)
 [0, 0, 0, 2]: 0.027199480682611465 (ACTUAL), 0.21759586036205292 (DESIRED)
 [0, 0, 0, 3]: 0.015308843925595284 (ACTUAL), 0.12247073650360107 (DESIRED)
 [0, 0, 0, 4]: 0.007754005491733551 (ACTUAL), 0.06203204393386841 (DESIRED)
Max absolute difference among violations: 5.4809346
Max relative difference among violations: 0.88978076
 ACTUAL: array([[[[ 1.739010e-02,  3.230270e-02,  2.719948e-02, ...,
           7.048368e-02,  7.131572e-02,  6.570359e-02],
         [-2.767790e-02, -4.453314e-02, -4.450583e-02, ...,...
 DESIRED: array([[[[ 1.391207e-01,  2.584216e-01,  2.175959e-01, ...,
           5.638694e-01,  5.705258e-01,  5.256287e-01],
         [-2.214233e-01, -3.562651e-01, -3.560466e-01, ...,...

In [22]:
ours[""]

array([[[[ 1.32819321e-02, -1.33312270e-01, -3.87987383e-02, ...,
           6.29887730e-02,  1.31967634e-01, -4.11989316e-02],
         [ 1.42699694e-02, -1.98177099e-01, -5.18712476e-02, ...,
           5.21292612e-02,  1.31137744e-01, -6.26785979e-02],
         [ 1.56879351e-02, -1.74667805e-01, -5.41146696e-02, ...,
           7.47463256e-02,  1.47957832e-01, -5.31596318e-02],
         ...,
         [-2.72745602e-02, -4.50189821e-02,  1.91823039e-02, ...,
           1.27687886e-01,  1.73087083e-02, -4.78906790e-03],
         [-3.38151976e-02, -3.48692648e-02,  2.08071694e-02, ...,
           1.33409724e-01,  2.58520171e-02, -6.89397752e-03],
         [-2.89378576e-02,  8.06565210e-03,  2.50202678e-02, ...,
           1.60044044e-01,  5.01406565e-02, -1.44538726e-03]],

        [[-8.42593759e-02, -1.06094234e-01, -5.77238463e-02, ...,
          -2.59142309e-01, -2.28232052e-02,  5.44510968e-02],
         [-9.94649008e-02, -1.29761487e-01, -1.40659571e-01, ...,
          -3.16927433e

In [23]:
theirs["/layers.0/self_attn/Transpose_1_output_0"]

array([[[[ 1.32819247e-02, -1.33312255e-01, -3.87987345e-02, ...,
           6.29887730e-02,  1.31967634e-01, -4.11989205e-02],
         [ 1.42699620e-02, -1.98177099e-01, -5.18712476e-02, ...,
           5.21292612e-02,  1.31137758e-01, -6.26786053e-02],
         [ 1.56879388e-02, -1.74667791e-01, -5.41146733e-02, ...,
           7.47463256e-02,  1.47957817e-01, -5.31596318e-02],
         ...,
         [-2.72745639e-02, -4.50189859e-02,  1.91823114e-02, ...,
           1.27687916e-01,  1.73087064e-02, -4.78906650e-03],
         [-3.38152088e-02, -3.48692574e-02,  2.08071694e-02, ...,
           1.33409739e-01,  2.58520059e-02, -6.89397566e-03],
         [-2.89378464e-02,  8.06562416e-03,  2.50202678e-02, ...,
           1.60044044e-01,  5.01406491e-02, -1.44538353e-03]],

        [[-8.42593834e-02, -1.06094219e-01, -5.77238277e-02, ...,
          -2.59142309e-01, -2.28232034e-02,  5.44511043e-02],
         [-9.94648933e-02, -1.29761487e-01, -1.40659571e-01, ...,
          -3.16927433e